# Stage 3 - Retrieve

## Data
Chroma DB

## Install
```

```

In [2]:
'''
1. Vectors DB Client
'''
import chromadb

client = chromadb.PersistentClient(path="chroma_kb") 
col = client.get_collection(name="knowledge_base")         

In [ ]:
'''
Test 1 search
Correct doc: #140
'''
from sentence_transformers import SentenceTransformer

QUERY = "боюсь, что ребёнок случайно купит фильм с моей карты"
e5 = SentenceTransformer("intfloat/multilingual-e5-base")
q_vec = e5.encode(["query: " + QUERY], normalize_embeddings=True)[0]

res = col.query(
  query_embeddings=[q_vec.tolist()],
  n_results=5,
)           

print(f"\nЗапрос: {QUERY}")
for doc, m, dist in zip(res["documents"][0], 
                        res["metadatas"][0],
                        res["distances"][0]):
  print(f"  dist={dist:.3f} [id: {m['id']}] [doc_id: {m['doc_id']}] [title: {m['title']}] {doc}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7252.17it/s]



Запрос: боюсь, что ребёнок случайно купит фильм с моей карты
  dist=0.162 [id: 123] [doc_id: 108] [title: Родительский контроль: обзор] Родительский контроль — общее название для настроек, которые ограничивают то, что ребёнок видит и делает в онлайн-кинотеатре. Почти везде он складывается из двух частей: фильтра по возрастным рейтингам и ограничения платных действий. Возрастной рейтинг присваивается фильму при выдаче прокатного удостоверения, сервис только показывает его в карточке и использует как фильтр. Рейтинг относится к фильму целиком, поэтому добрый на вид мультфильм с одной тревожной сценой может стоять на ступень выше, чем ожидают родители. Ограничение платных действий закрывает вторую типичную неприятность — оплату случайным нажатием. На телевизоре это особенно легко: карточка новинки и кнопка с ценой находятся в двух движениях пульта. Сервисы решают это по-разному: одни убирают раздел с покупками из детского режима, другие просят подтверждение перед оплатой.
  dist=0.169 [i

In [35]:
'''
Retrieve - dict
'''
from sentence_transformers import SentenceTransformer

THRESHOLD = 0.8

embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="mps")

def retrieve_dict(query, col=col, embedder=embedder, k=5, threshold=THRESHOLD):
  """Retrieval-пайплайн: embed query -> search -> filter -> return."""
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  res = col.query(query_embeddings=[q_vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append({
        'id': m['id'],
        'doc_id': m['doc_id'], 
        'similarity': round(sim, 4), 
        'text': doc, 
      }
    )
  return out

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7382.97it/s]


In [7]:
'''
Test: 3 queries (not gap)
'''
import json

# load queries
with open("1_queries.json", encoding="utf-8") as f:
  data = json.load(f)

# filter GAPs
queries = [u for u in data if u.get("kind") != "gap"]


for qi, q in enumerate(queries[:3]):
  ret = retrieve_dict(q["query"])
  print(f"\n[{qi}] {q['query']} {q['relevant_docs']}")
  if not ret:
    print("    (ничего выше порога — пайплайн вернул пусто)")
  for i in ret:
    print(f"    sim={i['similarity']:.3f} #{i['doc_id']} {i['text'][:58]} #{i['id']}")


[0] что означает ошибка P-102 [85]
    sim=0.861 #85 # Ошибка P-102: превышен лимит просмотров Ошибка P-102 поя #98
    sim=0.861 #178 Ошибка P-109 появляется, когда на устройстве не хватает св #207
    sim=0.858 #137 Ошибка P-106 появляется, когда плееру не удаётся загрузить #159
    sim=0.856 #138 Ошибка P-104 появляется, когда устройство не поддерживает  #160
    sim=0.856 #129 Ошибка P-105 появляется, когда на устройстве установлена у #150

[1] какая версия webOS или Tizen нужна для приложения на телевизоре [146]
    sim=0.871 #146 # На каких устройствах работает «Фильмоскоп» Телевизоры Sm #171
    sim=0.862 #86 # Поддерживаемые устройства На каких устройствах работает  #99
    sim=0.838 #129 Ошибка P-105 появляется, когда на устройстве установлена у #150
    sim=0.836 #172 Приложение на телевизоре обновляется само, если в магазине #199
    sim=0.836 #139 Если вы не знаете, к какой платформе относится ваш телевиз #161

[2] обязателен ли HDCP 2.2 для просмотра в 4K [110]
    sim=0.

In [8]:
'''
4.2 Test: compare with relevant docs
'''
# precision@K, recall@K и MRR

relevant = {qi: set(q["relevant_docs"]) for qi, q in enumerate(queries)}

problem = []
for qi, q in enumerate(queries):
    ret = retrieve_dict(q["query"])
    # hits = retrieve(q["query"], k=5, threshold=0.0)
    top_ids = [i['doc_id'] for i in ret]
    rel = relevant[qi]
    is_problem = (not top_ids) or (top_ids[0] not in rel)
    if is_problem:
        problem.append(qi)

print("Проблемные запросы:", problem)

for qi in problem:
    q = queries[qi]
    rel = relevant[qi]
    ret = retrieve_dict(q["query"], k=5)
    print(f"\n[{qi}] {q['query']}  (релевантные id: {sorted(rel) or 'нет'})")
    for i in ret:
        mark = "+" if i['doc_id'] in rel else " "
        print(f"    {mark} sim={i['similarity']:.3f} #{i['id']} {i['text'][:60]}")

# print
# print("\nЗапрос [8] без порога:",
#   [f"#{c}" for c, id, *_ in retrieve(queries[8]['query'], col, embedder,
#                                        k=3, threshold=0.0)])
# print("Запрос [8] с порогом 0.83:",
#   [f"#{c}" for c, id, *_ in retrieve(queries[8]['query'], col, embedder,
#                                        k=3, threshold=0.83)])

Проблемные запросы: [3, 6, 10, 13, 15, 18, 19, 20, 22, 23, 24, 25, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 41, 43, 44, 46, 47, 48, 50, 51, 53, 54]

[3] как подключить саундбар по HDMI ARC  (релевантные id: [22])
      sim=0.842 #145 Если при воспроизведении звук идет, а изображения нет — экра
      sim=0.834 #79 Объёмный звук дома собирают двумя способами, и выбор между н

[6] сколько стоит тариф Максимум и что в него входит  (релевантные id: [113])
      sim=0.855 #28 # Какой тариф подойдёт именно вам В «Фильмоскопе» три тарифа
    + sim=0.844 #132 # Тарифы «Фильмоскопа» Тариф выбирается при оформлении подпи
      sim=0.831 #37 Почему зрители выбирают «Максимум» «Максимум» — это кино без

[10] сколько профилей можно создать в одном аккаунте  (релевантные id: [63])
      sim=0.893 #72 Сколько профилей помещается в один аккаунт и как их создать,
    + sim=0.884 #73 # Профили в аккаунте Профиль — это отдельное пространство вн
      sim=0.868 #71 Аккаунт и профиль — не одно и то ж

In [36]:
import numpy as np

def recall_at_k(found, relevant, k=5):
  """Доля релевантных документов в первых k результатах."""
  hits = set(found[:k]) & relevant
  return len(hits) / len(relevant) if relevant else 0.0

rows = {"Обычный поиск": [], "Query expansion": [], "HyDE": []}

for q in queries:
    query, rel = q["query"], set(q["relevant_docs"])

    base = retrieve_dict(query)
    base_docs = [t['doc_id'] for t in base]
    rows["Обычный поиск"].append(recall_at_k(base_docs, rel))

n = len(queries)
print(f"{'Подход':<18}{'recall@5':>10}")
# for name in rows:
print(f"{"Обычный поиск":<18}{np.mean(rows["Обычный поиск"]):>10.3f}")

Подход              recall@5
Обычный поиск          0.723


In [1]:
def basic_search(query, k=5, threshold=0.8):
  ret = retrieve_dict(query, k=k, threshold=threshold)
  doc_ids = [i['doc_id'] for i in ret]
  return doc_ids

In [2]:
def precision_at_k(found_ids, relevant_ids, k=5):
  """Доля релевантных среди k выданных чанков."""
  hits = [i for i in found_ids[:k] if i in relevant_ids]
  return len(hits) / k


def reciprocal_rank(found_ids, relevant_ids):
    """Обратный ранг первого релевантного чанка; 0, если его нет в выдаче."""
    for rank, i in enumerate(found_ids, start=1):
        if i in relevant_ids:
            return 1.0 / rank
    return 0.0


K = 5
rows = []
for q in queries:
    rel = set(q["relevant_docs"])
    found = basic_search(q["query"])
    rows.append({
        "n_rel": len(rel),
        "p": precision_at_k(found, rel, K),
        "r": recall_at_k(found, rel, K),
        "rr": reciprocal_rank(found, rel),
    })

print('>>> Basic Search')
print(f"{'запрос':<44}{'|R|':>4}{'P@5':>7}{'R@5':>7}{'RR':>7}")
for row, q in zip(rows, queries):
    print(f"{q['query'][:42]:<44}{row['n_rel']:>4}"
          f"{row['p']:>7.2f}{row['r']:>7.2f}{row['rr']:>7.2f}")

print(f"\nprecision@5 = {np.mean([r['p'] for r in rows]):.3f}")
print(f"recall@5    = {np.mean([r['r'] for r in rows]):.3f}")
print(f"MRR         = {np.mean([r['rr'] for r in rows]):.3f}")

NameError: name 'queries' is not defined

In [32]:
for tr in [0.75, 0.8, 0.83]:
  for K in [1, 3, 5, 10, 20]:
    rows = []
    for q in queries:
      rel = set(q["relevant_docs"])
      found = basic_search(q["query"], k=K, threshold=tr)
      rows.append(recall_at_k(found, rel, K))

    print(f"K = {K} | threshold = {tr} | recall@5 = {np.mean(rows):.3f}")

K = 1 | threshold = 0.75 | recall@5 = 0.348
K = 3 | threshold = 0.75 | recall@5 = 0.670
K = 5 | threshold = 0.75 | recall@5 = 0.723
K = 10 | threshold = 0.75 | recall@5 = 0.911
K = 20 | threshold = 0.75 | recall@5 = 0.964
K = 1 | threshold = 0.8 | recall@5 = 0.348
K = 3 | threshold = 0.8 | recall@5 = 0.670
K = 5 | threshold = 0.8 | recall@5 = 0.723
K = 10 | threshold = 0.8 | recall@5 = 0.875
K = 20 | threshold = 0.8 | recall@5 = 0.929
K = 1 | threshold = 0.83 | recall@5 = 0.330
K = 3 | threshold = 0.83 | recall@5 = 0.616
K = 5 | threshold = 0.83 | recall@5 = 0.616
K = 10 | threshold = 0.83 | recall@5 = 0.688
K = 20 | threshold = 0.83 | recall@5 = 0.688


# Stage 3 - Report
- transformation is needed: Query expansion

Threshold = 0.8
K = 5